Step 1: Imports and Setup

In [1]:
# Standard libraries
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import shutil
import pandas as pd
import gc  # For memory management in batch processing

# Add parent directory to Python path to import from src/
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Image processing
import imageio.v2 as imageio  # avoids deprecation warnings
from IPython.display import Image, Video, display

# Your project-specific modules
from src.config import VOXEL_SIZE, ORGANELLES
from src.reconstruction import build_3d_scene, render_orbit_video
from src.utils import get_file_paths, get_centroid
from src.spatial_metrics import compute_spatial_metrics, find_roi_file
from src.tracking import run_tracking_pipeline, visualize_tracking
from src.metrics import compute_organelle_metrics

## Step 1: Configure Paths & Parameters
Configure these variables before running any analysis:

In [2]:
# ========================================
# CONFIGURE: Edit these paths as needed
# ========================================
stack_dir = "/Users/ariellerothman/Desktop/Full Sperm cell Stacks"  # Directory with TIFF stacks
output_dir = "/Users/ariellerothman/Desktop/Full Sperm cell Stacks"  # Where to save outputs
base_dir = stack_dir
sperm_id = 18  # Which sperm cell to analyze

os.makedirs(output_dir, exist_ok=True)

# Update organelle paths to this sperm cell stack
objects = []
for org in ORGANELLES:
    name = org["name"]
    new_org = org.copy()
    new_org["path"] = os.path.join(stack_dir, f"{name}_stack_1.tif")
    objects.append(new_org)

## Step 2: (Optional) Single-Cell Workflow Example
**Use this section to analyze ONE sperm cell in detail.**
(For batch processing multiple cells, skip to the batch processing section below.)

### Step 2a: Run Tracking Conversions

In [ ]:

# === Specify which sperm cell ===
# This converts TrackMate CSV files to long format for watershed segmentation
file_paths = get_file_paths(sperm_id=sperm_id, base_dir=stack_dir, registered=True)
cell_dir = base_dir

for organelle in ["MO", "mitochondria"]:
    print(f"\nProcessing tracking for: {organelle.upper()}")

    # Run pipeline: converts wide format CSV to long format for watershed (silently)
    long_csv_path = run_tracking_pipeline(
        cell_number=sperm_id,
        base_dir=base_dir,
        organelle=organelle,
        total_tracks=400,
        verbose=False
    )

    # Save to organized subfolder structure
    folder_map = {"MO": "MO tracking", "mitochondria": "Mito tracking"}
    tracking_dir = os.path.join(base_dir, f"Sperm {sperm_id}", folder_map[organelle])
    os.makedirs(tracking_dir, exist_ok=True)
    target_csv = os.path.join(tracking_dir, "temp_long.csv")
    shutil.move(long_csv_path, target_csv)
    
    print(f"Saved final long CSV to: {target_csv}")

# === Visualize only frames with tracks (save silently, no display) ===
tiff_path = file_paths["mitochondria"]  # path to the TIFF stack for this organelle
csv_path = target_csv                    # path to the just-saved CSV
visualize_tracking(tiff_path, csv_path, verbose=False)


### Step 2b: Compute All Organelle Metrics

In [ ]:
# === Set sperm cell ID ===
sample_id = f"sperm_{sperm_id}"
file_paths = get_file_paths(sperm_id=sperm_id, base_dir=base_dir, registered=True)

# === Log file paths for QC ===
from src.utils import log_file_paths
print(f"File Locations:")
for log_line in log_file_paths(sperm_id, file_paths, registered=True):
    print(log_line)

# === Output folder ===
output_dir = os.path.join(base_dir, f"Sperm {sperm_id}", "Organellar_Measures")
os.makedirs(output_dir, exist_ok=True)

# === Get reference centroids (for distance calculations) ===
pseudopod_centroid = get_centroid(file_paths["pseudopod"])
nucleus_centroid   = get_centroid(file_paths["nucleus"])

# === Find converted tracking CSVs (created in Step 2a) ===
# If Step 2a wasn't run, these will be None and metrics will use connected components
cell_dir = os.path.join(base_dir, f"Sperm {sperm_id}")
mito_csv_path = os.path.join(cell_dir, "Mito tracking", "temp_long.csv")
mo_csv_path = os.path.join(cell_dir, "MO tracking", "temp_long.csv")
mito_csv = mito_csv_path if os.path.exists(mito_csv_path) else None
mo_csv = mo_csv_path if os.path.exists(mo_csv_path) else None

# === Organelle input spec: (name, path_to_stack, optional_tracking_csv) ===
# Tracking CSVs are provided for mitochondria and MO (which have multiple instances)
# These are the converted long-format CSVs from Step 2a, or None if not run
# Other organelles (pseudopod, nucleus, sperm_cell) are single objects, so no CSV
organelle_inputs = [
    ("mitochondria", file_paths["mitochondria"], mito_csv),
    ("MO", file_paths["MO"], mo_csv),
    ("pseudopod", file_paths["pseudopod"], None),
    ("nucleus", file_paths["nucleus"], None),
    ("sperm_cell", file_paths["sperm_cell"], None),
]

# === Run all metrics and save ===
all_metrics = []
for organelle_name, stack_path, csv_path in organelle_inputs:
    df = compute_organelle_metrics(
        organelle_name, stack_path, csv_path,
        pseudopod_centroid, nucleus_centroid, sample_id
    )
    all_metrics.append(df)

full_df = pd.concat(all_metrics, ignore_index=True)
metrics_path = os.path.join(output_dir, f"{sample_id}_all_metrics.csv")
full_df.to_csv(metrics_path, index=False)

# === QC Summary ===
from src.metrics import get_metrics_summary, validate_metrics
print(f"\n Metrics Summary:")
summary = get_metrics_summary(full_df)
for key, val in summary.items():
    if key == "validation_warnings":
        if val:
            for warning in val:
                print(warning)
    else:
        print(f"  {key}: {val}")

# === Data Validation ===
validation_issues = validate_metrics(full_df)
if validation_issues:
    print(f"\n  Validation Issues:")
    for issue in validation_issues:
        print(f"  {issue}")
else:
    print(f"\n Data validation passed!")

print(f"\nMetrics saved to: {metrics_path}")


### Step 2c: Compute Spatial Metrics

Computes distances and angles relative to a reference point in the sample (e.g., spermatheca location).

In [ ]:
# ========================================
# CONFIGURE: Reference point (xyz coordinates in global image space)
# ========================================
# IMPORTANT: This reference point is defined in the FULL/GLOBAL image space
# The ROI file automatically converts sperm cell coordinates (from cropped images)
# back to global space, so they align with this reference point.
reference_point_xyz = [58, 1256, 2464]  # Spermathecal valve center [Z, Y, X] in voxels

paths = get_file_paths(sperm_id, base_dir, registered=True)
roi_path = find_roi_file(os.path.dirname(paths["sperm_cell"]))

# === Run computation ===
metrics = compute_spatial_metrics(
    sperm_cell_path=paths["sperm_cell"],
    pseudopod_path=paths["pseudopod"],
    nucleus_path=paths["nucleus"],
    roi_path=roi_path,
    reference_point_xyz=reference_point_xyz
)

# Convert metrics dict to DataFrame (one row)
metrics_df = pd.DataFrame([metrics])

# Define output path (inside Sperm cell folder)
excel_path = os.path.join(base_dir, f"Sperm {sperm_id}", "spatial_metrics.xlsx")

# Save to Excel
metrics_df.to_excel(excel_path, index=False)

print(f"Spatial metrics saved to: {excel_path}")

# Optional: display metrics
from pprint import pprint
pprint(metrics)

### Step 2d: Build 3D Reconstruction & Render Video

In [ ]:
os.makedirs(output_dir, exist_ok=True)

# === Get full file paths using utility (unregistered for 3D reconstruction) ===
file_paths = get_file_paths(sperm_id, base_dir, registered=False)

# === Build organelle input for reconstruction ===
objects = []
for org in ORGANELLES:
    path = file_paths[org["name"]]
    if os.path.exists(path):
        new_obj = org.copy()
        new_obj["path"] = path
        objects.append(new_obj)
    else:
        print(f"Missing file for {org['name']}: {path}")

# === Build 3D scene and render orbit video ===
output_dir = os.path.join(base_dir, f"Sperm {sperm_id}")
os.makedirs(output_dir, exist_ok=True)

try:
    print(f"Building 3D scene for Sperm {sperm_id}...")
    plotter = build_3d_scene(
        objects=objects,
        voxel_size=VOXEL_SIZE,
        sperm_mask_path=file_paths["sperm_cell"]  # Pass sperm cell mask path to restrict organelles
    )

    video_path = os.path.join(output_dir, f"Sperm_{sperm_id}_rotation.gif")
    print(f"Rendering orbit video (this may take 1-2 minutes)...")
    render_orbit_video(plotter, output_path=video_path)
    print(f"3D reconstruction video saved to: {video_path}")

    # === Display the video in notebook ===
    if os.path.exists(video_path):
        print(f"\nVideo preview:")
        display(Image(filename=video_path))
    else:
        print(f"Video file was not created at {video_path}")
        
except ValueError as e:
    print(f"\nVideo rendering failed: {str(e)}")
    print("   Troubleshooting tips:")
    print("   1. Check that all organelle TIFF files exist in the correct directory")
    print("   2. Verify that TIFF files contain valid binary mask data")
    print("   3. Try running the diagnostic cell to check file paths")
except Exception as e:
    print(f"\nUnexpected error during 3D reconstruction: {str(e)[:100]}")
    import traceback
    traceback.print_exc()

## Step 3: BATCH PROCESSING (Multiple Sperm Cells)
**Processes multiple sperm cells in one run.** This is the recommended workflow for analyzing many cells.

### How to Use Batch Processing:
1. **Edit the configuration** in the cell below (paths & sperm IDs)
2. **Run the cell** - it will process all cells and save outputs
3. **Check progress** - status messages show which cell is being processed
4. **View results** - Excel file contains all metrics for all cells

In [4]:
# ========================================
# CONFIGURE: Batch Processing Settings
# ========================================
parent_dir = "/Users/ariellerothman/Desktop/Full Sperm cell Stacks"
sperm_ids_to_process = [7]  # Edit this list: which sperm cells to process
output_dir = os.path.join(parent_dir, "batch_results_all_cells")
excel_path = os.path.join(output_dir, "all_metrics_batch.xlsx")

# Configuration from src.config (single source of truth for all parameters)
from src.config import (
    REFERENCE_POINT_XYZ,  # Spermathecal valve center in global image space
    TRACKING_BLOCK_SIZE,  # TrackMate block size
    MESH_MIN_SIZE,        # Min voxels for 3D mesh extraction
    MESH_BLUR,            # Gaussian blur on organelle stacks
    MESH_CLOSE_RADIUS,    # Morphological closing radius
)

In [ ]:

# ========================================
# RUN: Execute batch processing for all sperm cells
# ========================================
import glob as glob_module

os.makedirs(output_dir, exist_ok=True)

all_batch_metrics = []
processing_log = []

print(f" Starting batch processing for {len(sperm_ids_to_process)} sperm cells...")
print(f"   Output folder: {output_dir}\n")

for idx, sperm_id in enumerate(sperm_ids_to_process, 1):
    try:
        print(f"[{idx}/{len(sperm_ids_to_process)}] Processing Sperm {sperm_id}...", end=" ")
        
        # Get file paths with registered versions (for tracking conversion)
        file_paths = get_file_paths(sperm_id=sperm_id, base_dir=parent_dir, registered=True)
        cell_dir = os.path.join(parent_dir, f"Sperm {sperm_id}")
        
        # === CONVERT TRACKING CSVs (if not already converted) ===
        mo_csv = None
        mito_csv = None
        
        for organelle in ["MO", "mitochondria"]:
            folder_map = {"MO": "MO tracking", "mitochondria": "Mito tracking"}
            tracking_dir = os.path.join(cell_dir, folder_map[organelle])
            long_csv_path = os.path.join(tracking_dir, "temp_long.csv")
            
            if os.path.exists(long_csv_path):
                if organelle == "MO":
                    mo_csv = long_csv_path
                else:
                    mito_csv = long_csv_path
            else:
                try:
                    long_csv_output = run_tracking_pipeline(
                        cell_number=sperm_id,
                        base_dir=parent_dir,
                        organelle=organelle,
                        verbose=False
                    )
                    os.makedirs(tracking_dir, exist_ok=True)
                    shutil.move(long_csv_output, long_csv_path)
                    if organelle == "MO":
                        mo_csv = long_csv_path
                    else:
                        mito_csv = long_csv_path
                except Exception as e:
                    pass
        
        # === GENERATE TRACKING OVERLAYS ===
        try:
            for organelle in ["mitochondria", "MO"]:
                folder_map = {"MO": "MO tracking", "mitochondria": "Mito tracking"}
                tracking_dir = os.path.join(cell_dir, folder_map[organelle])
                long_csv_path = os.path.join(tracking_dir, "temp_long.csv")
                tiff_path = file_paths[organelle]
                
                if os.path.exists(tiff_path) and os.path.exists(long_csv_path):
                    try:
                        overlay_dir = visualize_tracking(
                            tiff_path=tiff_path,
                            csv_path=long_csv_path,
                            frames_to_display=200,
                            save_overlays=True,
                            verbose=False,
                            organelle_name=organelle
                        )
                    except Exception as e:
                        pass
        except Exception as e:
            pass
        
        # === Get reference centroids ===
        pseudopod_centroid = get_centroid(file_paths["pseudopod"])
        nucleus_centroid = get_centroid(file_paths["nucleus"])
        
        # === Compute organelle metrics ===
        organelle_inputs = [
            ("mitochondria", file_paths["mitochondria"], mito_csv),
            ("MO", file_paths["MO"], mo_csv),
            ("pseudopod", file_paths["pseudopod"], None),
            ("nucleus", file_paths["nucleus"], None),
            ("sperm_cell", file_paths["sperm_cell"], None),
        ]
        
        sample_id = f"sperm_{sperm_id}"
        cell_metrics = []
        for org_name, stack_path, csv_path in organelle_inputs:
            df = compute_organelle_metrics(
                org_name, stack_path, csv_path,
                pseudopod_centroid, nucleus_centroid, sample_id
            )
            cell_metrics.append(df)
        
        # Combine all organelles for this cell
        cell_df = pd.concat(cell_metrics, ignore_index=True)
        all_batch_metrics.append(cell_df)
        
        # === QC Summary for this cell ===
        from src.metrics import get_metrics_summary
        summary = get_metrics_summary(cell_df)
        qc_status = []
        qc_status.append(f"mito:{summary.get('mitochondria_count', 0)}")
        qc_status.append(f"mo:{summary.get('MO_count', 0)}")
        if "pseudopod_volume_um3" in summary:
            qc_status.append(f"pod:{summary['pseudopod_volume_um3']:.0f}µm³")
        if "nucleus_volume_um3" in summary:
            qc_status.append(f"nuc:{summary['nucleus_volume_um3']:.0f}µm³")
        print(f" QC({', '.join(qc_status)})", end="")
        
        # === Compute spatial metrics ===
        try:
            file_paths_crop = get_file_paths(sperm_id=sperm_id, base_dir=parent_dir, registered=True)
            roi_path = find_roi_file(os.path.dirname(file_paths_crop["sperm_cell"]))
            
            spatial_metrics = compute_spatial_metrics(
                sperm_cell_path=file_paths_crop["sperm_cell"],
                pseudopod_path=file_paths_crop["pseudopod"],
                nucleus_path=file_paths_crop["nucleus"],
                roi_path=roi_path,
                reference_point_xyz=REFERENCE_POINT_XYZ
            )
            print(f"  Metrics ({spatial_metrics['distance_centroid_to_target_um']:.1f}µm)", end="")
        except Exception as e:
            print(f"   Spatial metrics failed: {str(e)[:40]}", end="")
        
        # === Build 3D reconstruction and render video ===
        try:
            file_paths_3d = get_file_paths(sperm_id=sperm_id, base_dir=parent_dir, registered=False)
            
            objects_3d = []
            for org in ORGANELLES:
                path = file_paths_3d[org["name"]]
                if os.path.exists(path):
                    new_obj = org.copy()
                    new_obj["path"] = path
                    objects_3d.append(new_obj)
            
            cell_output_dir = os.path.join(cell_dir, "3D_Reconstruction")
            os.makedirs(cell_output_dir, exist_ok=True)
            
            plotter = build_3d_scene(
                objects=objects_3d,
                voxel_size=VOXEL_SIZE,
                sperm_mask_path=file_paths_3d["sperm_cell"]
            )
            
            video_path = os.path.join(cell_output_dir, f"Sperm_{sperm_id}_rotation.gif")
            render_orbit_video(plotter, output_path=video_path)
            print(f"  3D", end="")
        except ValueError as e:
            if "no meshes" in str(e).lower():
                print(f"   3D: no meshes", end="")
            else:
                print(f"   3D error: {str(e)[:40]}", end="")
        except Exception as e:
            print(f"   3D error: {str(e)[:40]}", end="")
        
        print()  # Newline at end
        processing_log.append({"Sperm_ID": sperm_id, "Status": " Success", "Distance_µm": spatial_metrics.get('distance_centroid_to_target_um')})
            
    except FileNotFoundError as e:
        print(f" Files not found: {str(e)[:60]}")
        processing_log.append({"Sperm_ID": sperm_id, "Status": f" Error: {str(e)[:40]}", "Distance_µm": None})
    except Exception as e:
        print(f" Error: {str(e)[:60]}")
        processing_log.append({"Sperm_ID": sperm_id, "Status": f" Error: {str(e)[:40]}", "Distance_µm": None})

# === Save combined results ===
if all_batch_metrics:
    batch_df = pd.concat(all_batch_metrics, ignore_index=True)
    batch_df.to_excel(excel_path, index=False)
    print(f"\n Batch processing complete!")
    print(f"    Results: {excel_path}")
    print(f"    Total metrics: {len(batch_df)} rows ({len(sperm_ids_to_process)} cells)")
    
    # Save processing log
    log_df = pd.DataFrame(processing_log)
    log_path = os.path.join(output_dir, "batch_processing_log.csv")
    log_df.to_csv(log_path, index=False)
    print(f"    Log: {log_path}")
    
    # Show where 3D videos are saved
    print(f"\n 3D Reconstruction Videos:")
    for sperm_id in sperm_ids_to_process:
        video_dir = os.path.join(parent_dir, f"Sperm {sperm_id}", "3D_Reconstruction")
        video_file = os.path.join(video_dir, f"Sperm_{sperm_id}_rotation.gif")
        if os.path.exists(video_file):
            print(f"   ✓ Sperm {sperm_id}: Video saved")
        else:
            print(f"     Sperm {sperm_id}: Video not found")
    
    # Show where tracking overlays are saved
    print(f"\n Tracking Overlay Images:")
    for sperm_id in sperm_ids_to_process:
        cell_dir = os.path.join(parent_dir, f"Sperm {sperm_id}")
        for organelle, folder_name in [("mitochondria", "Mito tracking"), ("MO", "MO tracking")]:
            overlay_dir = os.path.join(cell_dir, folder_name, "tracking_overlays", f"{organelle}_overlays")
            if os.path.exists(overlay_dir):
                overlay_count = len([f for f in os.listdir(overlay_dir) if f.endswith('.png')])
                if overlay_count > 0:
                    print(f"   ✓ Sperm {sperm_id} {organelle}: {overlay_count} overlay images")
            else:
                print(f"     Sperm {sperm_id} {organelle}: No overlays generated")
else:
    print(f"\n No metrics computed. Check file paths and configuration.")


 Starting batch processing for 1 sperm cells...
   Output folder: /Users/ariellerothman/Desktop/Full Sperm cell Stacks/batch_results_all_cells

[1/1] Processing Sperm 7...  QC(mito:108, mo:85, pod:24µm³, nuc:1µm³)  Metrics (12.5µm)Extracting meshes:
  mitochondria: 260413 → 182145 verts (69.9%), 365250 faces
  MO: 265853 → 186048 verts (70.0%), 372420 faces
  pseudopod: 187830 → 131481 verts (70.0%), 262958 faces
  nucleus: 14600 → 10221 verts (70.0%), 20430 faces
  sperm_cell: 222790 → 155953 verts (70.0%), 311902 faces

Successfully extracted 5 organelle meshes
🎥 Rendering 60-frame orbit video...
  Frame 15/60
  Frame 30/60
  Frame 45/60
  Frame 60/60
 Saving 60 frames to video...
Video saved successfully! (60 frames)
  3D

 Batch processing complete!
    Results: /Users/ariellerothman/Desktop/Full Sperm cell Stacks/batch_results_all_cells/all_metrics_batch.xlsx
    Total metrics: 196 rows (1 cells)
    Log: /Users/ariellerothman/Desktop/Full Sperm cell Stacks/batch_results_all_cells

## Step 4: Unfused MO Analysis (Optional)
**Separate analysis for unfused mitochondrial organelles when applicable.**

In [ ]:
# ========================================
# CONFIGURE: Unfused MO Batch Settings
# ========================================
parent_dir_unfused = "/Users/ariellerothman/Desktop/Full Sperm cell Stacks"
sperm_ids_unfused = [12]  # Which sperm cells have unfused MO data
output_excel = os.path.join(parent_dir_unfused, "unfused_mo_metrics.xlsx")
report_path  = os.path.join(parent_dir_unfused, "unfused_mo_run_report.csv")

# Import configuration from src.config (single source of truth)
from src.config import (
    ORGANELLE_THRESHOLD, VOXEL_SIZE, PIXEL_SIZE_UM,
    SLICE_THICKNESS_UM, VOXEL_VOLUME
)

# ========================================
# Helper Functions
# ========================================
def ci_listdir(pattern):
    """List files matching glob pattern."""
    return glob.glob(pattern)

def ci_filter(paths, *tokens):
    """Filter paths by multiple tokens (case-insensitive)."""
    toks = [t.lower() for t in tokens if t]
    out = []